# Planificación

La **planificación** encuentra una secuencia de acciones que alcanza un objetivo, como la [búsqueda](search.ipynb), pero los estados no son nodos de un grafo escrito de antemano. Están **descritos por hechos**, y las acciones por lo que requieren y lo que cambian. El planificador deduce por sí mismo qué acciones se pueden aplicar en cada estado y adónde llevan, así que el mismo programa planifica en cualquier dominio descrito de esta forma.

La formulación clásica es **STRIPS** (Richard Fikes y Nils Nilsson, 1971), escrita para Shakey, el primer robot móvil que planificaba sus propias acciones. Su descendiente, **PDDL** (1998), es el lenguaje estándar de los planificadores actuales.

## Estados y acciones como hechos

El **mundo de los bloques**: bloques sobre una mesa, apilados unos sobre otros, y una sola acción, mover un bloque que no tiene nada encima a la mesa o encima de otro bloque libre. Un **estado** es un conjunto de hechos como `("on", "C", "A")` (C está sobre A) y `("clear", "B")` (no hay nada sobre B). Una **acción** tiene tres conjuntos de hechos:

- **precondiciones**, que deben cumplirse para que la acción se pueda aplicar;
- una **lista de añadidos**, los hechos que hace verdaderos;
- una **lista de borrados**, los hechos que hace falsos.

In [ ]:
from itertools import product

blocks = ["A", "B", "C"]
places = blocks + ["table"]

actions = []
for block, origin, destination in product(blocks, places, places):
    if len({block, origin, destination}) < 3:
        continue
    destination_must_be_clear = {("clear", destination)} if destination != "table" else set()
    origin_becomes_clear = {("clear", origin)} if origin != "table" else set()
    actions.append({
        "name": f"move {block} from {origin} to {destination}",
        "preconditions": {("on", block, origin), ("clear", block)} | destination_must_be_clear,
        "add": {("on", block, destination)} | origin_becomes_clear,
        "delete": {("on", block, origin)} | destination_must_be_clear,
    })

print(len(actions), "actions, for example:", actions[0]["name"])
for part in ["preconditions", "add", "delete"]:
    print(f"  {part:13}", sorted(actions[0][part]))

18 actions, for example: move A from B to C
  preconditions [('clear', 'A'), ('clear', 'C'), ('on', 'A', 'B')]
  add           [('clear', 'B'), ('on', 'A', 'C')]
  delete        [('clear', 'C'), ('on', 'A', 'B')]


La mesa siempre está libre, así que mover un bloque a ella no necesita la precondición de que el destino esté libre. Aplicar una acción a un estado es aritmética de conjuntos: quitar la lista de borrados y después añadir la lista de añadidos.

In [ ]:
def applicable(action, state):
    return action["preconditions"] <= state


def apply(action, state):
    return frozenset((state - action["delete"]) | action["add"])

## La anomalía de Sussman

El inicio: C sobre A, con A y B sobre la mesa. El objetivo: A sobre B y B sobre C, una torre de A, B y C de arriba abajo.

Gerald Sussman señaló a principios de los 70 que este problema sencillo derrota a un planificador que aborda los objetivos de uno en uno. Poner primero A sobre B deja C todavía sobre A, y hay que volver a quitar A; poner primero B sobre C entierra A bajo la torre. Resolverlo exige intercalar los objetivos.

In [ ]:
start = frozenset({("on", "C", "A"), ("on", "A", "table"), ("on", "B", "table"), ("clear", "C"), ("clear", "B")})
goal = {("on", "A", "B"), ("on", "B", "C")}

Planificación como búsqueda en anchura sobre los estados: la frontera guarda estados con el plan que llegó a ellos, y un estado es objetivo cuando contiene todos los hechos del objetivo.

In [ ]:
from collections import deque


def plan_by_breadth_first_search(start, goal, actions):
    frontier = deque([(start, [])])
    visited = {start}
    while frontier:
        state, plan = frontier.popleft()
        if goal <= state:
            return plan, len(visited)
        for action in actions:
            if applicable(action, state):
                next_state = apply(action, state)
                if next_state not in visited:
                    visited.add(next_state)
                    frontier.append((next_state, plan + [action["name"]]))
    return None, len(visited)


plan, states_seen = plan_by_breadth_first_search(start, goal, actions)
for step, action_name in enumerate(plan, 1):
    print(step, action_name)
print("States seen:", states_seen)

1 move C from A to table
2 move B from table to C
3 move A from table to B
States seen: 13


La búsqueda encontró el plan más corto, tres movimientos, después de ver 13 estados: con tres bloques hay exactamente 13 formas de colocarlos, así que los vio todos.

## Planificación a gran escala

La búsqueda a ciegas no escala: el número de colocaciones crece de forma explosiva con el número de bloques. Los planificadores reales usan **heurísticas** calculadas a partir de las propias descripciones de las acciones. El planificador FF (Jörg Hoffmann y Bernhard Nebel, 2001), por ejemplo, estima la distancia al objetivo resolviendo un problema relajado que ignora todas las listas de borrados, un problema lo bastante fácil como para resolverlo en cada paso. La planificación gestiona la logística y los calendarios de fabricación, y ha volado al espacio: el Remote Agent de la NASA planificó y ejecutó las operaciones de la sonda Deep Space 1, de forma autónoma, en 1999. Los videojuegos también la usan: los enemigos de *F.E.A.R.* (2005) elegían sus tácticas con un planificador al estilo STRIPS.

Los agentes construidos sobre modelos de lenguaje se enfrentan al mismo problema, elegir una secuencia de llamadas a herramientas que alcance un objetivo, pero describen las acciones en lenguaje natural en lugar de con precondiciones y efectos, así que ganan flexibilidad y pierden las garantías del planificador.